# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/viki22uied/ML/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**Source:** the ML-08/reference-pipeline model (`random_forest`, client-holdout split, `is_declining_label`
target) joined to the reference pipeline's rule-based reason codes (`scripts/02_baseline_score.py`), on
the same 30,000-row starter dataset. The playbook ranks by the model's decline-risk probability
(`prob_random_forest`) and attaches the same human-readable reason codes the baseline rule already
computes (`stale_visible_page`, `declining_with_demand`, `thin_visible_page`, `page_one_decay_risk`,
`low_ctr_visible_page`, `low_engagement_visible_page`) so a reviewer never has to trust a bare probability
number — every row in the queue below carries the concrete signals behind its rank.


In [1]:
import sys
sys.path.append('../../scripts')
import numpy as np
import pandas as pd
from ml_utils import PROCESSED_DIR

predictions = pd.read_csv(PROCESSED_DIR / "model_predictions.csv")
baseline_queue = pd.read_csv(PROCESSED_DIR / "baseline_refresh_queue.csv")

queue = predictions.merge(
    baseline_queue[["content_id", "reason_codes", "impressions_90d", "avg_position", "ctr",
                     "content_age_days", "days_since_last_update", "trend_direction"]],
    on="content_id", how="left",
)

queue["decline_risk_score"] = queue["prob_random_forest"].round(4)
queue = queue.sort_values("decline_risk_score", ascending=False).reset_index(drop=True)
queue["action_rank"] = np.arange(1, len(queue) + 1)

display_cols = ["action_rank", "content_id", "decline_risk_score", "reason_codes",
                 "impressions_90d", "avg_position", "ctr", "trend_direction", "is_declining_label"]
top_25 = queue[display_cols].head(25)
print(f"Total items ranked: {len(queue):,}")
print(f"Top-25 preview (of the ranked action queue):")
print(top_25.to_string(index=False))

print()
print("Reason code frequency across the top 200 highest-risk items:")
top200_reasons = queue.head(200)["reason_codes"].str.split("|").explode().value_counts()
print(top200_reasons)

print()
declining_rate_top50 = queue.head(50)["is_declining_label"].mean()
declining_rate_all = queue["is_declining_label"].mean()
print(f"Declining-label rate in top-50 of the queue: {declining_rate_top50:.3f} "
      f"vs. base rate across all items: {declining_rate_all:.3f} "
      f"(lift = {declining_rate_top50 / declining_rate_all:.2f}x)")


Total items ranked: 30,000
Top-25 preview (of the ranked action queue):
 action_rank           content_id  decline_risk_score                                                           reason_codes  impressions_90d  avg_position  ctr trend_direction  is_declining_label
           1 content_6eeb07cca243              0.8518                      declining_with_demand|low_engagement_visible_page            24516          26.9 0.09            down                   1
           2 content_f552433bab3c              0.8511                                                  declining_with_demand              337           1.8 0.00            down                   1
           3 content_d6570c51c9bd              0.8498                             declining_with_demand|low_ctr_visible_page             2498          10.1 0.00            down                   1
           4 content_816431f05a3b              0.8489                                                  declining_with_demand             192

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who:** a content/SEO editor or reviewer at a FlyRank client, using the ranked queue as a weekly or
monthly triage list — "which pages should I look at first this cycle." **What it's for:** prioritizing
limited editorial attention across hundreds or thousands of pages, not deciding *what edit* to make on
any single page (the reason codes point at a category — stale, thin, low CTR — not a specific fix).

**Where it stops being valid:**
- **Build-window bound.** The model was trained and evaluated on one dataset/label build. A new content
  mix, a new client onboarding, or a seasonal shift in search behavior is outside what was measured —
  the ranking should be refreshed on a regular cadence, not treated as permanent.
- **Association, not causation.** `decline_risk_score` says "this page's profile resembles pages that
  were already declining" — it does not say refreshing the page *will* reverse that (see the w06
  methodology questions on the paper's own refresh-lift claims: the same caveat applies to model scores).
- **No client-specific tuning.** One global model across all clients in the training data; a client with
  an unusual content strategy (e.g. mostly video landing pages, or a niche with structurally low CTR)
  may see systematically mis-ranked items — this is a known blind spot, not a hidden one.
- **Not a publishing or de-indexing decision.** The score supports *review*, never an automatic content
  change, takedown, or reallocation of budget without a human reading the actual page.


In [2]:
print('Discussion-only section -- see markdown above.')

Discussion-only section -- see markdown above.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any item in the queue, a human must:**
- Open the actual page and confirm the reason code still matches reality (e.g. `thin_visible_page` on a
  page that was already updated last week is a stale signal, not a live one).
- Check for context the model cannot see: seasonality, a planned campaign, legal/compliance content that
  must not be touched, or a page intentionally kept thin (e.g. a hub/landing page by design).
- Confirm the client relationship allows this kind of change without prior sign-off.

**Never automate directly from this output:**
- Auto-publishing rewritten content, or auto-deleting/de-indexing a "declining" page, from the score
  alone — the score is decision-support, not an execution trigger.
- Treating the top-N list as a strict order for staffing decisions or performance reviews of content
  writers — it ranks pages, not people, and using it that way is a misuse of the tool.
- Any action on a page flagged for legal, medical, financial, or other regulated content categories
  without the relevant compliance review, regardless of its risk score.


In [3]:
print('Discussion-only section -- see markdown above.')

Discussion-only section -- see markdown above.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Retrain/refresh the model when any of these hold:**
- **Time-based:** more than one quarter since the last build — the paper's own lifecycle findings (w06,
  Finding #2) show portfolio health shifts materially across ~90-day windows, so a static model risks
  drifting out of date on the same horizon its own subject matter changes.
- **Population drift:** the client mix, content-type mix, or average `content_age_days` in the live
  population diverges noticeably (e.g. >15–20% relative shift) from the training population's summary
  stats computed below — a model trained on one shape of portfolio degrades silently on another.
- **Score calibration drift:** the declining-rate lift in the top-50 of a fresh scoring run
  (recomputed the same way as section 1) drops meaningfully below the lift measured here — a sign the
  ranking is no longer separating signal from noise the way it did at build time.
- **Base-rate shift:** the overall `is_declining_label` rate moves a lot from the ~54% measured here —
  a large market-wide shift (e.g. an algorithm update affecting many clients at once) is exactly the
  kind of out-of-distribution event that should force a re-look, not a routine retrain.


In [4]:
import sys
sys.path.append('../../scripts')
import pandas as pd
from ml_utils import PROCESSED_DIR

feature_vector = pd.read_csv(PROCESSED_DIR / "refresh_feature_vector.csv")

monitoring_snapshot = {
    "n_rows": int(len(feature_vector)),
    "n_clients": int(feature_vector["client_id"].nunique()),
    "declining_label_rate": round(float(feature_vector["is_declining_label"].mean()), 4),
    "avg_content_age_days": round(float(feature_vector["content_age_days"].mean()), 1),
    "median_content_age_days": round(float(feature_vector["content_age_days"].median()), 1),
    "avg_days_since_last_update": round(float(feature_vector["days_since_last_update"].mean()), 1),
    "content_type_mix": feature_vector["content_type"].value_counts(normalize=True).round(4).to_dict(),
}
for k, v in monitoring_snapshot.items():
    print(f"{k}: {v}")

import json, os
os.makedirs("../outputs", exist_ok=True)
with open("../outputs/w07_monitoring_snapshot.json", "w") as f:
    json.dump(monitoring_snapshot, f, indent=2)
print()
print("Snapshot written to work/outputs/w07_monitoring_snapshot.json -- compare a future run's snapshot "
      "against this one to detect the population drift described above.")


n_rows: 30000
n_clients: 32
declining_label_rate: 0.5421
avg_content_age_days: 256.2
median_content_age_days: 236.0
avg_days_since_last_update: 46.1
content_type_mix: {'keyword article': 0.9069, 'feedly article': 0.0699, 'comparison article': 0.0232}

Snapshot written to work/outputs/w07_monitoring_snapshot.json -- compare a future run's snapshot against this one to detect the population drift described above.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*


In [5]:
import sys
sys.path.append('../../scripts')
import numpy as np
import pandas as pd
from ml_utils import PROCESSED_DIR

predictions = pd.read_csv(PROCESSED_DIR / "model_predictions.csv")
baseline_queue = pd.read_csv(PROCESSED_DIR / "baseline_refresh_queue.csv")

queue = predictions.merge(
    baseline_queue[["content_id", "reason_codes", "impressions_90d", "avg_position", "ctr",
                     "content_age_days", "days_since_last_update", "trend_direction"]],
    on="content_id", how="left",
)
queue["decline_risk_score"] = queue["prob_random_forest"].round(4)
queue = queue.sort_values("decline_risk_score", ascending=False).reset_index(drop=True)
queue["action_rank"] = np.arange(1, len(queue) + 1)

export_cols = ["action_rank", "content_id", "client_id", "decline_risk_score", "reason_codes",
               "impressions_90d", "avg_position", "ctr", "trend_direction", "is_declining_label"]

import os
os.makedirs("../outputs", exist_ok=True)
queue[export_cols].head(500).to_csv("../outputs/w07_action_playbook_queue_top500.csv", index=False)

top200_reasons = queue.head(200)["reason_codes"].str.split("|").explode().value_counts()
top200_reasons.to_csv("../outputs/w07_top_reason_codes.csv", header=["count"])

print("Wrote work/outputs/w07_action_playbook_queue_top500.csv "
      f"({min(500, len(queue))} rows) and work/outputs/w07_top_reason_codes.csv")
print()
print(top200_reasons)


Wrote work/outputs/w07_action_playbook_queue_top500.csv (500 rows) and work/outputs/w07_top_reason_codes.csv

reason_codes
declining_with_demand          198
low_ctr_visible_page            58
low_engagement_visible_page     47
page_one_decay_risk              2
general_refresh_review           2
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere — only pseudonymized `client_id`/`content_id`
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
